# Tokenizer Playground

1. 用 BERT 观察特殊 token：`[UNK]` / `[CLS]` / `[SEP]` / `[PAD]`
2. 对比 Qwen 与 mBERT，并观察 `padding_side`

> BERT 没有名为 BOS/EOS 的 token，对应的是 `[CLS]` / `[SEP]`。

In [ ]:
import os

# 避免 HF Hub / 公司代理把 kernel 卡死；模型已在本地缓存
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased", local_files_only=True)

print("special tokens:", tokenizer.special_tokens_map)
print(
    "pad / cls / sep / unk:",
    tokenizer.pad_token,
    tokenizer.cls_token,
    tokenizer.sep_token,
    tokenizer.unk_token,
)

## 1. 特殊 Token 观察

### UNK

词表外字符（如 emoji）会落到 `[UNK]`。

In [ ]:
unk_tokens = tokenizer.tokenize("hello 💩")
print("UNK 示例:", unk_tokens)

### CLS / SEP（对应 BOS / EOS）

- `tokenize()` 只切词，**不加**特殊 token
- `tokenizer()` / `encode()` **默认会加** `[CLS]` 和 `[SEP]`

In [ ]:
print("only tokenize:", tokenizer.tokenize("hello world"))

encoded = tokenizer("hello world")
print("CLS/SEP tokens:", tokenizer.convert_ids_to_tokens(encoded["input_ids"]))
print("CLS/SEP ids   :", encoded["input_ids"])

### PAD

batch 里短句补齐到最长句长度时才会出现 `[PAD]`，并用 `attention_mask` 标出有效位。

In [ ]:
batch = tokenizer(
    ["hi", "hello world this is longer"],
    padding=True,
)

print("短句 tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][0]))
print("短句 ids   :", batch["input_ids"][0])
print("attention_mask:", batch["attention_mask"][0])
print()
print("长句 tokens:", tokenizer.convert_ids_to_tokens(batch["input_ids"][1]))
print("长句 ids   :", batch["input_ids"][1])
print("attention_mask:", batch["attention_mask"][1])

## 2. Tokenizer 对比实验

- **tokenizer_a**: `Qwen/Qwen2.5-0.5B-Instruct`（BPE）
- **tokenizer_b**: `bert-base-multilingual-cased`（WordPiece）

### 准备

先加载两个 tokenizer，再定义测试文本。

In [1]:
from transformers import AutoTokenizer

tokenizer_a = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct", local_files_only=True
)
tokenizer_b = AutoTokenizer.from_pretrained(
    "bert-base-multilingual-cased", local_files_only=True
)

print("tokenizer_a:", tokenizer_a.name_or_path, "vocab_size=", tokenizer_a.vocab_size)
print("tokenizer_b:", tokenizer_b.name_or_path, "vocab_size=", tokenizer_b.vocab_size)

/Users/bytedance/Desktop/ai_learn/transformer-inference-lab/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] The tokenizer you are loading from 'bert-base-multilingual-cased' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


tokenizer_a: Qwen/Qwen2.5-0.5B-Instruct vocab_size= 151643
tokenizer_b: bert-base-multilingual-cased vocab_size= 119547


测试文本：

In [2]:
texts = [
    "你好，世界！",
    "Transformer inference is interesting.",
    "大模型 inference optimization",
    "KV Cache 可以减少重复计算。",
    "Hello世界",
    "RTX 5070 Ti",
    "print('hello world')",
    "3.1415926",
    "🙂🚀🔥",
    "  前后有空格  ",
]

### 实验 1：同一句话的 token 数对比

对相同文本分别 tokenize，比较 token 序列与长度差异。

In [3]:
def inspect_tokenizer(name, tokenizer, text):
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.convert_tokens_to_ids(tokens)
    print(f"{name}:", tokens)
    print(f"{name}_ids:", ids)
    print(f"{name}_length:", len(ids))
    print(f"{name}_decode:", tokenizer.decode(ids))


for item in texts:
    print("original text:", item)
    inspect_tokenizer("tokenizer_a", tokenizer_a, item)
    inspect_tokenizer("tokenizer_b", tokenizer_b, item)
    print("-" * 100)

original text: 你好，世界！
tokenizer_a: ['ä½łå¥½', 'ï¼Į', 'ä¸ĸçķĮ', 'ï¼ģ']
tokenizer_a_ids: [108386, 3837, 99489, 6313]
tokenizer_a_length: 4
tokenizer_a_decode: 你好，世界！
tokenizer_b: ['你', '好', '，', '世', '界', '！']
tokenizer_b_ids: [2262, 3240, 10064, 2087, 5621, 10055]
tokenizer_b_length: 6
tokenizer_b_decode: 你 好 ， 世 界 ！
----------------------------------------------------------------------------------------------------
original text: Transformer inference is interesting.
tokenizer_a: ['Transformer', 'Ġinference', 'Ġis', 'Ġinteresting', '.']
tokenizer_a_ids: [46358, 44378, 374, 7040, 13]
tokenizer_a_length: 5
tokenizer_a_decode: Transformer inference is interesting.
tokenizer_b: ['Trans', '##former', 'in', '##ference', 'is', 'interesting', '.']
tokenizer_b_ids: [29608, 59106, 10106, 52790, 10124, 64888, 119]
tokenizer_b_length: 7
tokenizer_b_decode: Transformer inference is interesting.
----------------------------------------------------------------------------------------------------
orig

### 实验 2：`padding_side` left vs right

对同一 batch 分别设置 `padding_side="left"` / `"right"`，打印 `input_ids` 与 `attention_mask`。

**结论：** Padding 改变批处理中的 token 位置和 attention mask，但不应改变原始文本的有效 token。

In [ ]:
batch_texts = ["hi", "hello world this is longer"]


def show_padding(side: str):
    # 用 Qwen 演示；生成式模型推理时常见 left padding
    tokenizer_a.padding_side = side
    batch = tokenizer_a(batch_texts, padding=True)
    print(f"===== padding_side={side!r} =====")
    for i, text in enumerate(batch_texts):
        ids = batch["input_ids"][i]
        mask = batch["attention_mask"][i]
        tokens = tokenizer_a.convert_ids_to_tokens(ids)
        valid_ids = [tid for tid, m in zip(ids, mask) if m == 1]
        print(f"text: {text!r}")
        print("tokens        :", tokens)
        print("input_ids     :", ids)
        print("attention_mask:", mask)
        print("valid_ids     :", valid_ids)
        print("valid_decode  :", tokenizer_a.decode(valid_ids))
        print()


show_padding("right")
show_padding("left")

# 复原，避免影响后续 cell
tokenizer_a.padding_side = "right"

核对：左右 padding 下，`attention_mask == 1` 对应的有效 token 应完全一致。

In [ ]:
# 验证：左右 padding 后，有效 token（mask==1）应一致
right = tokenizer_a(batch_texts, padding=True)
tokenizer_a.padding_side = "left"
left = tokenizer_a(batch_texts, padding=True)
tokenizer_a.padding_side = "right"

for i, text in enumerate(batch_texts):
    right_valid = [tid for tid, m in zip(right["input_ids"][i], right["attention_mask"][i]) if m == 1]
    left_valid = [tid for tid, m in zip(left["input_ids"][i], left["attention_mask"][i]) if m == 1]
    same = right_valid == left_valid
    print(f"{text!r}: valid tokens identical? {same}")
    print("  right valid:", right_valid)
    print("  left  valid:", left_valid)